# 02 · Biblioteca de platillos y menús

**Fase CRISP-DM:** 2. Entendimiento de los datos

**Objetivo:** definir **qué platillos existen** (biblioteca canónica) y **qué ofrece cada restaurante** a lo largo del año (menús con precios, ingredientes, alérgenos, altas, bajas y agotados).

| Entradas | Salidas (`data/synthetic/<run_id>/`) |
|---|---|
| `data/external/canonical_dishes.yaml`, `restaurants.parquet` (nb 01), Censo 2020 (escolaridad por AGEB) | `canonical_dishes`, `canonical_dish_aliases`, `menu_items`, `menu_item_attributes`, `menu_item_price_history`, `menu_item_availability_events` + `ground_truth/` |

> **Supuesto declarado (S1):** no existe una fuente pública con menús; la biblioteca y los menús son **sintéticos**. Lo que sí es real: los restaurantes (DENUE) y el nivel socioeconómico de su zona (Censo).

## Configuración

In [1]:
import numpy as np
import pandas as pd

from puzzle_recsys.catalog.cuisine_imputation import CuisineRules, extract_brand
from puzzle_recsys.catalog.library import load_library
from puzzle_recsys.config import CONFIG_DIR, load_paths, load_simulation
from puzzle_recsys.repository.parquet_repo import run_id_for, start_run
from puzzle_recsys.simulation.distributions import make_rng
from puzzle_recsys.simulation.menus import build_menus, evolve_menus
from puzzle_recsys.sources.inegi import read_ageb_indicators

paths, sim = load_paths(), load_simulation()
rng = make_rng(sim.seed, "menus")  # flujo aleatorio propio de esta etapa
library_file = paths.external / "canonical_dishes.yaml"
restaurants = pd.read_parquet(paths.interim / "world" / "restaurants.parquet")
print(f"Restaurantes: {len(restaurants):,} · meses: {sim.scale.months}")

Restaurantes: 1,500 · meses: 12


## 1. Biblioteca canónica

Escrita a mano y **validada al cargar** (pydantic): un ingrediente inexistente, un alérgeno desconocido o un slug duplicado detienen la ejecución. Los **alérgenos y las etiquetas de dieta no se escriben por platillo**: se derivan de los ingredientes, así no pueden contradecirse.

In [2]:
library = load_library(library_file)
frames = library.to_frames()
dishes = frames["canonical_dishes"]
print(
    f"{len(dishes)} platillos · {dishes['cuisine'].nunique()} cocinas · "
    f"{len(frames['canonical_dish_aliases'])} alias · {dishes['is_shareable'].sum()} compartibles"
)
pd.crosstab(dishes["cuisine"], dishes["course"], margins=True)

220 platillos · 21 cocinas · 653 alias · 50 compartibles


course,DESSERT,DRINK,MAIN,SIDE,STARTER,All
cuisine,,,,,,
antojitos,0,1,10,0,2,13
argentina_cortes,0,1,4,1,3,9
cafeteria,2,7,6,0,1,16
china,0,0,6,1,2,9
espanola,1,1,4,0,5,11
familiar_cadena,1,1,5,0,1,8
francesa,2,0,4,0,1,7
general,4,9,0,0,0,13
hamburguesas,0,1,5,2,0,8


In [3]:
allergen_freq = dishes["allergens"].explode().value_counts(normalize=True).rename("% de platillos")
diet_freq = dishes["diet_tags"].explode().value_counts(normalize=True).rename("% de platillos")
pd.concat([allergen_freq, diet_freq]).round(3).to_frame()

,% de platillos
lacteos,0.314
gluten,0.282
huevo,0.128
soya,0.058
ajonjoli,0.045
crustaceos,0.042
frutos_secos,0.038
pescado,0.035
sulfitos,0.022
cacahuate,0.019


## 2. Nivel socioeconómico de la zona

El precio de un restaurante depende de dónde está. Como *proxy* real se usa el **grado promedio de escolaridad** (`GRAPROES`) de su AGEB en el Censo 2020, estandarizado (`ses_z`, 0 = promedio de la ciudad). Las AGEB sin dato toman la mediana.

In [4]:
ses = read_ageb_indicators(paths.raw / "ageb_mza_urbana_09_cpv2020_csv.zip", ["GRAPROES"])
ses = ses.set_index("cvegeo")["graproes"]
graproes = restaurants["ageb_cvegeo"].map(ses)
print(f"Restaurantes sin dato de escolaridad: {graproes.isna().sum()}")
restaurants["ses_z"] = (graproes.fillna(ses.median()) - ses.mean()) / ses.std()
restaurants.groupby("alcaldia")["ses_z"].mean().sort_values().round(2).to_frame("ses_z promedio").T

Restaurantes sin dato de escolaridad: 9


alcaldia,Venustiano Carranza,Iztapalapa,Milpa Alta,Tláhuac,Xochimilco,Iztacalco,La Magdalena Contreras,Gustavo A. Madero,Azcapotzalco,Tlalpan,Álvaro Obregón,Cuajimalpa de Morelos,Cuauhtémoc,Coyoacán,Miguel Hidalgo,Benito Juárez
ses_z promedio,-1.38,-0.66,-0.6,-0.44,-0.43,0.03,0.07,0.29,0.29,0.52,0.75,0.79,0.9,1.15,1.22,1.51


### Cadenas: un solo menú para todas sus sucursales

Las sucursales de una cadena (Vips, Toks, Sanborns…) comparten **la misma carta, los mismos nombres, los mismos precios y los mismos cambios**, igual que una `MenuTemplate` en Puzzle. La marca se detecta con las reglas no transferibles del notebook 01. El tamaño del menú depende del personal ocupado (DENUE).

In [5]:
rules = CuisineRules.from_yaml(paths.external / "cuisine_rules.yaml")
restaurants["chain"] = extract_brand(restaurants["name"], rules)
restaurants["chain"].value_counts().rename("sucursales").to_frame().T

chain,toks,vips,wings,sanborns,lyni,ihop,california,potzollcalli
sucursales,23,6,5,3,1,1,1,1


## 3. Menús iniciales

Cada menú mezcla platillos de su cocina, del pool general (bebidas y postres comunes) y un ~8 % de otras cocinas. Precio = precio típico × multiplicador **lognormal** del restaurante (centrado según `ses_z`) × ruido lognormal por platillo, con redondeo psicológico (…0, …5, …9).

In [6]:
menus = build_menus(restaurants, library, sim.supply, rng)
items = menus["menu_items"].merge(menus["ground_truth_menu_items"], on="menu_item_id")
per_menu = items.groupby("restaurant_id").size().median()
print(f"{len(items):,} platillos · {per_menu:.0f} por menú (mediana)")
prices = items.assign(precio_mxn=items["price_cents"] / 100)
prices.groupby("course")["precio_mxn"].describe(percentiles=[0.1, 0.5, 0.9]).round(0)

27,068 platillos · 18 por menú (mediana)


,count,mean,std,min,10%,50%,90%,max
course,,,,,,,,
DESSERT,3197.0,77.0,24.0,10.0,49.0,75.0,109.0,220.0
DRINK,6647.0,71.0,38.0,15.0,35.0,59.0,130.0,265.0
MAIN,12576.0,154.0,98.0,15.0,75.0,135.0,245.0,1475.0
SIDE,833.0,53.0,21.0,19.0,35.0,49.0,75.0,170.0
STARTER,3815.0,112.0,56.0,15.0,55.0,100.0,189.0,520.0


In [7]:
pricing = menus["ground_truth_restaurant_pricing"].merge(restaurants[["restaurant_id", "ses_z"]])
pricing["quintil_ses"] = pd.qcut(
    pricing["ses_z"], 5, labels=["Q1 (bajo)", "Q2", "Q3", "Q4", "Q5 (alto)"]
)
pricing.groupby("quintil_ses", observed=True)["price_multiplier"].median().round(3).to_frame(
    "multiplicador mediano"
).T

quintil_ses,Q1 (bajo),Q2,Q3,Q4,Q5 (alto)
multiplicador mediano,0.89,1.046,1.136,1.181,1.247


## 4. Nombres en la carta

La carta rara vez usa el nombre canónico. El ruido es intencional: es lo que la **homologación (notebook 09)** tiene que deshacer, y su verdad queda en `ground_truth`.

In [8]:
display(items["name_variant"].value_counts(normalize=True).round(3).to_frame("proporción").T)
items.groupby("name_variant").sample(3, random_state=sim.seed)[
    ["name", "canonical_true", "name_variant"]
]

name_variant,CANONICAL,ALIAS,DECORATED,TYPO
proporción,0.397,0.363,0.179,0.061


,name,canonical_true,name_variant
17087,Guacamole con totopos,guacamole,ALIAS
13952,TACOS DE CAMARÓN CON QUESO,tacos_gobernador,ALIAS
907,JUMUS,hummus,ALIAS
13535,Torta cubana,torta_cubana,CANONICAL
13499,Michelada,michelada,CANONICAL
2785,Enchiladas suizas,enchiladas_suizas,CANONICAL
11248,Frijoles de la olla de la casa,frijoles_refritos,DECORATED
23506,Chocolate abuelita tradicional,chocolate_caliente,DECORATED
21381,Torta de mila de la casa,torta_milanesa,DECORATED
10789,Agua de horhcata,horchata,TYPO


## 5. Evolución del menú en el tiempo

Cada mes: altas, bajas y aumentos de precio; cada día, algunos platillos se agotan y se reponen a la mañana siguiente. Sin este historial **no se puede reconstruir qué había en el menú en cada visita**, que son los negativos del modelo (ADR 0003).

In [9]:
evolved = evolve_menus(
    menus, restaurants, library, sim.supply, sim.calendar, months=sim.scale.months, rng=rng
)
summary = {
    "platillos al inicio": len(menus["menu_items"]),
    "platillos en el año": len(evolved["menu_items"]),
    "dados de baja": int(evolved["menu_items"]["archived_at"].notna().sum()),
    "cambios de precio": len(evolved["menu_item_price_history"]) - len(evolved["menu_items"]),
}
pd.Series(summary).to_frame("conteo").T

,platillos al inicio,platillos en el año,dados de baja,cambios de precio
conteo,27068,36115,6265,14386


In [10]:
evolved["menu_item_availability_events"]["reason"].value_counts().to_frame("eventos").T

reason,SOLD_OUT,RESTOCKED,CREATED,ARCHIVED
eventos,206486,206486,36115,6265


### Verificaciones de integridad

Se comprueban en los datos generados las reglas que **antes se violaban** (y que ahora cubren tests de regresión):

1. Ningún cambio de precio ocurre después de que el platillo se dio de baja.
2. Todo nombre marcado como error de dedo (`TYPO`) es distinto del canónico.
3. Todas las sucursales de una cadena tienen la misma carta vigente al final del año.

In [11]:
from puzzle_recsys.catalog.cuisine_imputation import normalize_name

ph = evolved["menu_item_price_history"].merge(
    evolved["menu_items"][["menu_item_id", "archived_at"]], on="menu_item_id"
)
items_all = evolved["menu_items"].merge(evolved["ground_truth_menu_items"], on="menu_item_id")
canon_name = items_all["canonical_true"].map(dishes.set_index("canonical_id")["name"])
typos = items_all["name_variant"] == "TYPO"
active = items_all[items_all["archived_at"].isna()].merge(
    restaurants[["restaurant_id", "chain"]], on="restaurant_id"
)
menus_per_chain = (
    active.dropna(subset=["chain"])
    .groupby(["chain", "restaurant_id"])["canonical_true"]
    .apply(frozenset)
    .groupby("chain")
    .nunique()
)
checks = {
    "precios después de la baja": int((ph["valid_from"] > ph["archived_at"]).sum()),
    "TYPO idéntico al canónico": int(
        (
            items_all.loc[typos, "name"].map(normalize_name)
            == canon_name[typos].map(normalize_name)
        ).sum()
    ),
    "cadenas con cartas distintas entre sucursales": int((menus_per_chain > 1).sum()),
}
assert all(v == 0 for v in checks.values()), checks
pd.Series(checks, name="violaciones").to_frame().T

,precios después de la baja,TYPO idéntico al canónico,cadenas con cartas distintas entre sucursales
violaciones,0,0,0


## 6. Guardado de la corrida

Las tablas del **contrato** van a `data/synthetic/<run_id>/` y la **verdad oculta** a `ground_truth/`. Se guarda también una copia de la configuración usada.

In [12]:
repo = start_run(
    paths.synthetic,
    run_id_for(sim.run_name, sim.seed),
    config_files=[
        CONFIG_DIR / "simulation.yaml",
        library_file,
        paths.external / "cuisine_rules.yaml",
    ],
)
contract = {
    "restaurants": restaurants.drop(columns=["ses_z"]),
    "canonical_dishes": dishes,
    "canonical_dish_aliases": frames["canonical_dish_aliases"],
    "menu_items": evolved["menu_items"],
    "menu_item_attributes": evolved["menu_item_attributes"],
    "menu_item_price_history": evolved["menu_item_price_history"],
    "menu_item_availability_events": evolved["menu_item_availability_events"],
}
for name, table in contract.items():
    repo.write_table(name, table)
repo.ground_truth.write_table("menu_items", evolved["ground_truth_menu_items"])
repo.ground_truth.write_table("restaurant_pricing", evolved["ground_truth_restaurant_pricing"])
repo.ground_truth.write_table("restaurant_ses", restaurants[["restaurant_id", "ses_z"]])
print(repo.root)
pd.Series({n: len(t) for n, t in contract.items()}, name="filas").to_frame().T

C:\Users\natal\Cundinamarca\Puzzle\Recomendador\data\synthetic\base-20261116


,restaurants,canonical_dishes,canonical_dish_aliases,menu_items,menu_item_attributes,menu_item_price_history,menu_item_availability_events
filas,1500,220,653,36115,233967,50501,455352


## Conclusiones

- **Biblioteca:** 220 platillos canónicos en 20 cocinas + pool general, con alérgenos y dietas **derivados** de los ingredientes. La validación atrapó errores reales del catálogo (ingredientes inexistentes, alias ambiguos entre platillos, cocinas con muy pocos platos fuertes).
- **Precios anclados a datos reales:** el multiplicador de precio crece con la escolaridad promedio del AGEB (proxy socioeconómico del Censo 2020).
- **Ruido controlado:** ~60 % de los nombres en carta difieren del canónico; la verdad queda en `ground_truth` para medir la homologación del notebook 09.
- **Historial completo:** altas, bajas, cambios de precio y agotados permiten reconstruir el menú disponible **a la fecha** de cada visita.
- **Pasa al notebook 03:** menús y su historial para simular visitas y pedidos.